# 02 · ETL Failure Lab — recover safely after failure

**Goal:** Implement replay-safe processing, reject invalid records and test re-ingestion.

**Setup:** Python + SQLite, with synthetic data only.

**CS bridge:** Uniqueness, state, idempotency and transactions.

**Caveat:** End-to-end exactly-once is a strong cross-system guarantee; this exercise demonstrates an idempotent local sink only.

## Watch and learn — API ingestion with dlt

[**Watch the original video on YouTube**](https://www.youtube.com/watch?v=5eMytPBgmVs). Suggested: watch a focused 5–15 minute segment.

[![Video thumbnail](https://img.youtube.com/vi/5eMytPBgmVs/hqdefault.jpg)](https://www.youtube.com/watch?v=5eMytPBgmVs)

Jupyter/VS Code may embed the video with internet access. GitHub preview may not; the code exercises do not depend on video playback.

In [1]:
# Optional video player (does not download or store the video)
try:
    from IPython.display import display, YouTubeVideo
    display(YouTubeVideo("5eMytPBgmVs", width=720, height=405))
except ImportError:
    print("Open the YouTube link in the Markdown above.")

**Additional resource:** [DataTalks.Club workshop — ingestion and incremental loading](https://github.com/DataTalksClub/data-engineering-zoomcamp/tree/main/cohorts/2026/workshops). Read the material before or after the lab, not instead of writing code.

## 1. Opening question

A pipeline loads three events successfully, then crashes just before reporting success. The scheduler retries. Will six rows exist in the target table?

In [2]:
import sqlite3
incoming=[
    {"event_id":"e1","magnitude":3.1},
    {"event_id":"e2","magnitude":4.2},
    {"event_id":"e3","magnitude":2.4},
    {"event_id":"bad","magnitude":None},
]
conn=sqlite3.connect(":memory:")
conn.execute("CREATE TABLE events(event_id TEXT PRIMARY KEY,magnitude REAL NOT NULL)")

## 2. Validate → Quarantine → Upsert

For this exercise, require a nonempty event ID and a numeric magnitude. Record a safe rejection reason instead of silently dropping malformed data.

In [3]:
def ingest(batch):
    bad=[]; accepted=[]
    for r in batch:
        if not r.get("event_id") or not isinstance(r.get("magnitude"),(int,float)):
            bad.append({"record":r,"reason":"invalid event_id or magnitude"})
        else:
            accepted.append((r["event_id"],r["magnitude"]))
    with conn:
        conn.executemany("""INSERT INTO events(event_id,magnitude) VALUES (?,?)
            ON CONFLICT(event_id) DO UPDATE SET magnitude=excluded.magnitude""",accepted)
    return {"accepted":len(accepted),"rejected":bad}
first=ingest(incoming)
rows_after_first=conn.execute("SELECT COUNT(*) FROM events").fetchone()[0]
second=ingest(incoming)
rows_after_replay=conn.execute("SELECT COUNT(*) FROM events").fetchone()[0]
print(first,"count after 1st:",rows_after_first,"after retry:",rows_after_replay)
assert rows_after_first == rows_after_replay == 3
assert len(first["rejected"]) == 1

{'accepted': 3, 'rejected': [{'record': {'event_id': 'bad', 'magnitude': None}, 'reason': 'invalid event_id or magnitude'}]} count after 1st: 3 after retry: 3


## 3. Late updates and versioning

Imagine `e2` arrives again with magnitude 4.0. This simple example overwrites the value. Should you also preserve historical versions, `source_updated_at` and an audit trail? The answer depends on the data contract.

In [4]:
ingest([{"event_id":"e2","magnitude":4.0}])
observed=conn.execute("SELECT magnitude FROM events WHERE event_id='e2'").fetchone()[0]
print("latest stored magnitude:",observed)
assert observed == 4.0

latest stored magnitude: 4.0


## 4. Transfer questions

- What if two updates to `e2` arrive in reverse order? Use `source_updated_at` to prevent an old value from winning.
- How would you make a batch of 100,000 records transactional?
- How many retries are appropriate after HTTP 429, and how do you identify a poison record?
- Why does `ON CONFLICT` alone not guarantee **end-to-end exactly-once processing**?

**Extension:** Add an `ingestion_runs` table with `started_at`, `finished_at` and `rejected_count`.

**Next:** [Stage 03](../curriculum/modules/03.md) and the [PostgreSQL/Supabase module](../curriculum/modules/04.md).